<a href="https://colab.research.google.com/github/shanto736/digital-marketing/blob/main/notebook0967cfcb59.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Business Objective**


The aim of this project is to predict which  ***customers will convert*** (make a purchase or take the desired action) from digital marketing campaigns.
This will help companies focus on the right customers and use their marketing budget more effectively.


# About
The dataset used in this project is the [Digital Marketing Campaign Dataset ](https://www.kaggle.com/datasets/rabieelkharoua/predict-conversion-in-digital-marketing-dataset/data)from Kaggle.


The dataset consists of 8,000 rows and 20 columns

Features

Demographic Information
* CustomerID: Unique identifier for each customer.
* Age: Age of the customer.Gender: Gender of the customer (Male/Female).
* Income: Annual income of the customer in USD.Marketing-specific


VariablesCampaignChannel: The channel through which the marketing campaign is delivered (Email, Social Media, SEO, PPC, Referral).

* CampaignType: Type of the marketing campaign (Awareness, Consideration, Conversion, Retention).
* AdSpend: Amount spent on the marketing campaign in USD.ClickThroughRate: Rate at which customers click on the marketing content.
* ConversionRate: Rate at which clicks convert to desired actions (e.g., purchases).
* AdvertisingPlatform: Confidential.AdvertisingTool: Confidential.


Customer Engagement Variables

* WebsiteVisits: Number of visits to the website.
* PagesPerVisit: Average number of pages visited per session.
* TimeOnSite: Average time spent on the website per visit (in minutes).
* SocialShares: Number of times the marketing content was shared on social media.
* EmailOpens: Number of times marketing emails were opened.
* EmailClicks: Number of times links in marketing emails were clicked.

Historical Data

* PreviousPurchases: Number of previous purchases made by the customer.
* LoyaltyPoints: Number of loyalty points accumulated by the customer.

Target Variable
* Conversion: Binary variable indicating whether the customer converted (1) or not (0).




# Data Exploration

**2.1 Load the Data**

Load the dataset


In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

df = pd.read_csv("/kaggle/input/predict-conversion-in-digital-marketing-dataset/digital_marketing_campaign_dataset.csv",header=0)
df


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/predict-conversion-in-digital-marketing-dataset/digital_marketing_campaign_dataset.csv'

# Display the first 5 rows of the dataset

In [ ]:
df.head()

# 2.2 Initial Data Inspection

**Inspecting the Dataset**


Use df.shape, df.info(), and df.describe() to inspect the dataset's size, data types, and summary statistics:



In [ ]:
df.shape


In [ ]:
df.info()


In [ ]:
df.describe()


# 2.3 Check for Duplicate Values



In [ ]:
df.duplicated().sum()


# 2.4 Column Names in the Dataset



In [ ]:
df.columns


# 2.5 Check for Missing Values


In [ ]:
df.isnull().sum()


This checks for the number of missing values in each feature to determine how to handle them, such as imputation or removing incomplete rows.

Result: No missing values were found in the dataset.

# Summary of Data Exploration

From the initial data exploration, we found that:

No missing values: This allows us to proceed with the analysis without needing to handle missing data.
No duplicate values: This ensures the dataset is of high quality for building models.
The dataset is well-structured and contains relevant features for the project's objectives.
The next step is to perform deeper analysis, such as exploring relationships between features and preparing the data for building Machine Learning models.



# 3. Data Cleaning
Since the CustomerID feature is merely a customer identifier and has no relevance to the analysis or Machine Learning model,


In [ ]:
del_col = ['CustomerID']
df_clean = df.drop(del_col, axis=1)

In [ ]:
df = df_clean
df_clean.columns

# Channel and campaign type distribution

In [ ]:
import matplotlib.pyplot as plt

# Channel and campaign type distribution
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

df['CampaignChannel'].value_counts().plot(kind='bar', ax=axes[0,0], title='Campaign Channels')
df['CampaignType'].value_counts().plot(kind='bar', ax=axes[0,1], title='Campaign Types')
df['Gender'].value_counts().plot(kind='pie', ax=axes[1,0], title='Gender Distribution', autopct='%1.1f%%')

# Conversion by channel
conv_by_channel = df.groupby('CampaignChannel')['Conversion'].agg(['count', 'sum', 'mean'])
conv_by_channel['conversion_rate'] = conv_by_channel['mean']
conv_by_channel['conversion_rate'].plot(kind='bar', ax=axes[1,1], title='Conversion Rate by Channel')

plt.tight_layout()
plt.show()

print("=== CONVERSION BY CHANNEL ===")
print(conv_by_channel)


# Spend and performance analysis


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Ad spend distribution
df['AdSpend'].hist(bins=20, ax=axes[0,0])
axes[0,0].set_title('Ad Spend Distribution')

# CTR vs Conversion Rate
df.plot.scatter(x='ClickThroughRate', y='ConversionRate', ax=axes[0,1], alpha=0.7)
axes[0,1].set_title('CTR vs Conversion Rate')

# Age vs Income
df.plot.scatter(x='Age', y='Income', c='Conversion', colormap='viridis', ax=axes[1,0])
axes[1,0].set_title('Age vs Income (colored by Conversion)')

# Engagement metrics
engagement_cols = ['WebsiteVisits', 'SocialShares', 'EmailOpens']
df[engagement_cols].boxplot(ax=axes[1,1])
axes[1,1].set_title('Engagement Metrics')

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='CampaignChannel', y='ConversionRate', data=df, palette='viridis')
plt.title('Distribution of Conversion Rate by Campaign Channel', fontsize=16)
plt.xlabel('Campaign Channel', fontsize=14)
plt.ylabel('Conversion Rate', fontsize=14)
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt



numeric_columns = df.select_dtypes(include=['int64', 'float64'])


correlation_matrix = numeric_columns.corr()


print(correlation_matrix)
plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix of All Numeric Columns', fontsize=16)
plt.show()
print(correlation_matrix['ConversionRate'].sort_values(ascending=False))


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
X = df[['AdSpend', 'ClickThroughRate', 'WebsiteVisits', 'PagesPerVisit', 'TimeOnSite']]
y= df['Conversion']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
model = LogisticRegression()
model.fit(X_train_scaled, y_train)
train_score = model.score(X_train_scaled, y_train)
test_score = model.score(X_test_scaled, y_test)

print("=== MMM MODEL PERFORMANCE ===")
print(f"Training accuracy: {train_score:.3f}")
print(f"Test accuracy: {test_score:.3f}")


In [ ]:
print("Feature Importance")

for feature, coef in zip(X.columns, model.coef_[0]):
    print(f"{feature}: {coef}")


In [ ]:
import matplotlib.pyplot as plt

# Plot feature importance
features = ['AdSpend', 'ClickThroughRate', 'WebsiteVisits', 'PagesPerVisit', 'TimeOnSite']
importance = [0.4035, 0.4266, 0.2959, 0.3295, 0.4247]

plt.barh(features, importance, color='skyblue')
plt.xlabel('Importance')
plt.title('Feature Importance for Conversion Prediction')
plt.show()


In [ ]:
from sklearn.model_selection import train_test_split
X, Xt, y, yt = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
[sum(y==1), sum(y==0), sum(yt==1), sum(yt==0)]


In [ ]:
from sklearn.preprocessing import MinMaxScaler
scl = MinMaxScaler().fit(X)
X = scl.transform(X)
Xt = scl.transform(Xt)


In [ ]:
from sklearn import metrics
def perf_eval(y_test, y_prob):
    yp = np.round(y_prob[:,1])
    ACC = metrics.accuracy_score(y_test, yp)
    REC = metrics.recall_score(y_test, yp)
    PRE = metrics.precision_score(y_test, yp)
    MCC = metrics.matthews_corrcoef(y_test, yp)
    F1 = metrics.f1_score(y_test, yp)
    AUC = metrics.roc_auc_score(y_test, y_prob[:,1])
    return [ACC, MCC, AUC, PRE, REC, F1]


In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_predict
param = [1,2]
clf = SVC(random_state=0, probability=True)
clf = GridSearchCV(estimator= clf, param_grid={'C': param }, cv=10).fit(X,y)
yp = cross_val_predict(clf.best_estimator_, X, y, cv=10, method = 'predict_proba')
perf_eval(y,yp)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(random_state=0)
param = {'n_estimators': [50, 100], 'max_depth': [5, 10]}
clf = GridSearchCV(estimator=clf, param_grid=param, cv=10).fit(X, y)
yp = cross_val_predict(clf.best_estimator_, X, y, cv=10, method='predict_proba')
perf_eval(y,yp)
